In [31]:
import pandas as pd

In [32]:
parts = [
    "../data/split/part_1.csv",
    "../data/split/part_2.csv",
    "../data/split/part_3.csv",
    "../data/split/part_4.csv"
]
samples = []
for part in parts:
    sample = pd.read_csv(part, usecols=["owner", "name", "combined_text"])
    sample = sample.sample(frac=0.25,random_state=42)#Changing the df size to 1M beacuse of i dont have enough memory
    samples.append(sample)
df = pd.concat(samples,ignore_index=True)

In [33]:
df.shape

(1044276, 3)

In [34]:
#If i run the cosine  on this data it will be very big and impossible to do beacuse of it can jump upto TB size.

In [35]:
#creating the vector data of this df

In [36]:
import joblib
vectorizer = joblib.load('../data/models/tfidf_vectorizer.pkl')
data_matrix = vectorizer.transform(df['combined_text'])  #only this column goes into TF-IDF

In [37]:
#So we got vector data on the full dataset

In [38]:
data_matrix.nnz

11703351

In [39]:
data_matrix.shape

(1044276, 115158)

In [40]:
data_matrix.dtype

dtype('float64')

In [41]:
print(f"Memory usage: {data_matrix.data.nbytes / 1e6:.2f} MB")

Memory usage: 93.63 MB


In [42]:
#TF-IDF are higher dimensional
#TruncatedSVD reduces the dimension and make it dense so the FAISS can work efficently

In [55]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=500,random_state=42)
reduced_matrix = svd.fit_transform(data_matrix)

In [51]:
reduced_matrix.shape

(1044276, 300)

In [59]:
print(f"Explained variance: {svd.explained_variance_ratio_.sum():.4f}") #This tells you how much information were captured into the compressed version


Explained variance: 0.3999


In [ ]:
#In here using dimension = 100 only gave as 24%. so not good

#I changed the dataset size to 1M 

#Now we are getting upto 40% is good

In [61]:
import psutil
print(f"Total RAM: {psutil.virtual_memory().total / 1e9:.2f} GB")
print(f"Available RAM: {psutil.virtual_memory().available / 1e9:.2f} GB")

Total RAM: 16.85 GB
Available RAM: 7.84 GB


In [62]:
joblib.dump(svd,"../data/models/svd_model.pkl")
joblib.dump(reduced_matrix,"../data/models/reduced_matrix.pkl")

['../data/models/reduced_matrix.pkl']

In [3]:
import joblib
reduced_matrix = joblib.load("../data/models/reduced_matrix.pkl")

In [9]:
import faiss
import numpy as np

vectors = reduced_matrix.astype(np.float32)
faiss.normalize_L2(vectors)

In [10]:
d = vectors.shape[1]
nlist = 1000

quantizer = faiss.IndexFlatIP(d)
index = faiss.IndexIVFFlat(quantizer,d,nlist,faiss.METRIC_INNER_PRODUCT)

In [11]:
index.train(vectors)
index.add(vectors)

In [12]:
index.nprobe = 10